In [2]:
import tac_cumulated_activity
import ab_dose_177lu
import SimpleITK as sitk
from pathlib import Path
import sys
import numpy as np
sys.path.append("C:\\ac225\\Ac-225")
import functions
from SimpleITK import ResampleImageFilter, Transform
import matplotlib.pyplot as plt
import csv
import pandas as pd
%load_ext autoreload
%autoreload 2

In [2]:
def read_resample(seg_path:str, recon_path:str):
    def read_dicom(path):
        image= functions.read_dicom(path)
                #print(image.GetSize())
                #print(image.GetSpacing())
        return image
            
    image_recon= read_dicom(recon_path)
    image_seg= read_dicom(seg_path)  

    def resample_seg(image_recon, image_seg):
        resampler = ResampleImageFilter()
        resampler.SetReferenceImage(image_recon)
        resampler.SetInterpolator(sitk.sitkNearestNeighbor)
        resampler.SetTransform(Transform())
        resampled_seg_img = resampler.Execute(image_seg)
        return resampled_seg_img 
    resampled_seg= resample_seg(image_recon, image_seg) 
    return sitk.GetArrayFromImage(resampled_seg)

def quantification(volume: np.array, seg: np.array, tumour_key:int):
    mask = seg == tumour_key
    nvox = mask.sum()
    mean = volume[mask].mean()
    total = mean * nvox
    return total

In [4]:
patients={"p11":0, "p15":1, "p24":2, "p32":3, "p33":4, "p17":5, "p23":6, "p25":7, "p28":8, "p31":9, "p13":10, "p12":11}
tumours= {"p24":[11,12], "p25":[11, 12], "p15":[11, 12, 13, 14, 22, 23, 24, 25, 26, 27, 28], "p33":[11, 12], 
		   "p32":[11, 12, 13, 21, 22], "p23":[11, 12, 13, 14,22, 24, 25, 26], "p17":[11, 12, 13, 14, 22, 24, 25, 26],
		   "p31":[11, 12,13, 14, 22, 23, 24, 25, 26, 27], "p28":[11, 12, 13, 14, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31], 
           "p11":[23, 25, 26, 27, 28, 29], "p13":[11, 12], "p12":[11, 12, 13, 14]}


max_t = max(len(v) for v in tumours.values())      # most tumours in any patient
table_rows = []
for patient in patients.keys():
	dict_patient={}
	spect_4h_path= f"F:\\champalimaud\\Ac225\\data_patients\\{patient}\\spects\\spect_04h\\spect_calib.nii"
	spect_24h_path= f"F:\\champalimaud\\Ac225\\data_patients\\{patient}\\spects\\spect_24h\\spect_calib.nii"

	if patient== "p12":
		spect_4h_path=None
		spect_24h_path= f"F:\\champalimaud\\Ac225\\data_patients\\{patient}\\spects\\spect_24h\\spect_calib.nii"
		spect_7d_path=  f"F:\\champalimaud\\Ac225\\data_patients\\{patient}\\spects\\spect_48h\\spect_calib.nii"

	else:
		def get_spect_tp3_path(patient, filename="spect_calib.nii"):
			spects_dir = Path(r"F:\champalimaud\Ac225\data_patients") / str(patient) / "spects"
			matches = sorted(spects_dir.glob(f"spect_*d/{filename}"))
			return matches[0]

		spect_7d_path = get_spect_tp3_path(patient)

	paths=[spect_4h_path, spect_24h_path, spect_7d_path]
	results = {t: [np.nan] * 3 for t in tumours[patient]}
	seg_path= f"F:\champalimaud\Ac225\patients_phantoms\{patient}\segmentations\seg_spect_24h_w_tumours.nrrd"
	seg_orig= sitk.GetArrayFromImage(functions.read_dicom(seg_path))
	for tumour in tumours[patient]:
		tumour_patient=[]
		for path in paths:
			if path is None:
				tumour_patient.append(np.nan)
				continue
			spect_arr= sitk.GetArrayFromImage(functions.read_dicom(path))
			seg_arr = seg_orig if seg_orig.shape == spect_arr.shape else read_resample(str(seg_path), str(path))
			result= quantification(spect_arr, seg_arr, tumour)
			results[tumour][paths.index(path)] = result
			tumour_patient.append(result)
		dict_patient[tumour]= tumour_patient
	print(f"patient {patient}: {dict_patient}")

	flat = [v for t in tumours[patient] for v in results[t]]   # t1,t2,t3 per tumour, in list order
	flat += [np.nan] * (3 * max_t - len(flat))                 # pad patients with fewer tumours
	table_rows.append([patient] + flat)

header1 = [""] + [f"tumour {k+1}" if j == 0 else "" for k in range(max_t) for j in range(3)]
header2 = [""] + ["t1", "t2", "t3"] * max_t

with open("F:\\champalimaud\\tese_artigo\\quantification_tumours_original.csv", "w", newline="") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(header1)
    w.writerow(header2)
    for row in table_rows:
        w.writerow(["" if (isinstance(x, float) and np.isnan(x)) else
                    (str(x) if not isinstance(x, str) else x)
                    for x in row])

patient p11: {23: [0.34394700289703906, 2.4271260164678097, 0.4830416990444064], 25: [0.47512943018227816, 1.3114132937043905, 0.24704249761998653], 26: [0.9666379946283996, 1.9514289954677224, 0.3513635324779898], 27: [0.16914581134915352, 0.3526485711336136, 0.024911028798669577], 28: [0.09631741850171238, 0.727318525314331, 0.12533788150176406], 29: [0.5773826844524592, 3.3588730171322823, 0.9999112263321877]}
patient p15: {11: [117.59515395015478, 208.89338716864586, 68.30339225381613], 12: [2.8575992807745934, 4.898175749927759, 0.15581989422207698], 13: [7.830503396689892, 18.011866807937622, 0.9351056055165827], 14: [7.4198268465697765, 12.92318880558014, 1.8098364099860191], 22: [5.8972271624952555, 12.582806423306465, 2.0506065599620342], 23: [0.46804554760456085, 2.347332179546356, 0.009873613016679883], 24: [8.849891506135464, 21.240944266319275, 2.165487999096513], 25: [1.3110870942473412, 3.6460433900356293, 0.43922826834023], 26: [0.715433869510889, 2.7743678092956543, 0.

In [4]:
patients={"p11":0, "p15":1, "p24":2, "p32":3, "p33":4, "p17":5, "p23":6, "p25":7, "p28":8, "p31":9, "p13":10, "p12":11}
tumours= {"p24":[11,12], "p25":[11, 12], "p15":[11, 12, 13, 14, 22, 23, 24, 25, 26, 27, 28], "p33":[11, 12], 
		   "p32":[11, 12, 13, 21, 22], "p23":[11, 12, 13, 14,22, 24, 25, 26], "p17":[11, 12, 13, 14, 22, 24, 25, 26],
		   "p31":[11, 12,13, 14, 22, 23, 24, 25, 26, 27], "p28":[11, 12, 13, 14, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31], 
           "p11":[23, 25, 26, 27, 28, 29], "p13":[11, 12], "p12":[11, 12, 13, 14]}

patients={"p11":0, "p15":1, "p24":2, "p32":3, "p33":4, "p17":5, "p23":6, "p25":7, "p28":8, "p31":9, "p13":10, "p12":11}
max_t = max(len(v) for v in tumours.values())      # most tumours in any patient
table_rows = []
for patient in patients.keys():
	dict_patient={}
	spect_24h_path= f"F:\\champalimaud\\Ac225\\patients_phantoms\{patient}\\recon_volumes_1\\lu177\\{patient}_lu177.nii"

	paths=[spect_24h_path]
	results = {t: [np.nan] * 1 for t in tumours[patient]}
	seg_path= f"F:\champalimaud\Ac225\patients_phantoms\{patient}\\recon_volumes\seg_spect_24h_w_tumours_resampled.nrrd"
	seg_orig= sitk.GetArrayFromImage(functions.read_dicom(seg_path))
	for tumour in tumours[patient]:
		tumour_patient=[]
		for path in paths:
			spect_arr= sitk.GetArrayFromImage(functions.read_dicom(path))
			seg_arr = seg_orig if seg_orig.shape == spect_arr.shape else read_resample(str(seg_path), str(path))
			result= quantification(spect_arr, seg_arr, tumour)
			results[tumour][paths.index(path)] = result
			tumour_patient.append(result)
		dict_patient[tumour]= tumour_patient
	print(f"patient {patient}: {dict_patient}")

	flat = [v for t in tumours[patient] for v in results[t]]   # t1,t2,t3 per tumour, in list order
	flat += [np.nan] * (3 * max_t - len(flat))                 # pad patients with fewer tumours
	table_rows.append([patient] + flat)

header1 = [""] + [f"tumour {k+1}" if j == 0 else "" for k in range(max_t) for j in range(3)]
header2 = [""] + ["t1", "t2", "t3"] * max_t

with open("F:\\champalimaud\\tese_artigo\\quantification_tumours.csv", "w", newline="") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(header1)
    w.writerow(header2)
    for row in table_rows:
        w.writerow(["" if (isinstance(x, float) and np.isnan(x)) else
                    (str(x) if not isinstance(x, str) else x)
                    for x in row])

patient p11: {23: [1.1369885257445276], 25: [0.5246502086520195], 26: [1.2366716600954533], 27: [0.1496791047975421], 28: [0.35592876793816686], 29: [1.8842623326927423]}
patient p15: {11: [284.75018329918385], 12: [4.510355114936829], 13: [17.45182480663061], 14: [14.900512721389532], 22: [10.339382462203503], 23: [1.249538365751505], 24: [22.683154731988907], 25: [1.5514188706874847], 26: [1.4940962344408035], 27: [1.1889333054423332], 28: [2.964573785662651]}
patient p24: {11: [19.798340424895287], 12: [21.736151792109013]}
patient p32: {11: [21.15382321551442], 12: [38.517371237277985], 13: [4.959454629570246], 21: [0.6654930766671896], 22: [1.3865967709571123]}
patient p33: {11: [15.527580946683884], 12: [5.807426750659943]}
patient p17: {11: [21.226859986782074], 12: [3.2448456287384033], 13: [158.96720230579376], 14: [0.0], 22: [655.1384035348892], 24: [9.551045961678028], 25: [0.2553303533932194], 26: [1.463286533609498e-09]}
patient p23: {11: [111.98391401767731], 12: [610.128

dosimetry run

In [5]:
true_lk= {"p24": 57.1, "p25":95.2, "p33":59.0 ,"p15":68.8, "p23":88.3, "p17":23.5, "p32":25.9, "p28":47.8, "p31":53.8, "p13":40.5, "p12":49.0, "p11": 68.1565}
true_rk= {"p24": 60.3, "p25":116.1, "p33":50.1 , "p15":69.3, "p23":93.4, "p17":52.8, "p32":30.3, "p28":46.5, "p31":63.7, "p13":51.3, "p12":39.8, "p11":52.8016}

patients={"p11":0, "p15":1, "p24":2, "p32":3, "p33":4, "p17":5, "p23":6, "p25":7,  "p28":8, "p31":9, "p13":10, "p12":11}

tumours= {"p24":[11,12], "p25":[11, 12], "p15":[11, 12, 13, 14, 22, 23, 24, 25, 26, 27, 28], "p33":[11, 12], 
		   "p32":[11, 12, 13, 21, 22], "p23":[11, 12, 13, 14,22, 24, 25, 26], "p17":[11, 12, 13, 14, 22, 24, 25, 26],
		   "p31":[11, 12,13, 14, 22, 23, 24, 25, 26, 27], "p28":[11, 12, 13, 14, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31], 
           "p11":[23, 25, 26, 27, 28, 29], "p13":[11, 12], "p12":[11, 12, 13, 14]}

true_dosimetry= False
table = []
all_lesions = []
for patient in patients.keys():
		patient_id= patients[patient]
		tia_rk_lu, tia_lk_lu, tia_liver, tia_spleen, tia_bkg, tia_tumours_lu= tac_cumulated_activity.main(patient_id, true_dosimetry)

		#Lu-177 dose computation
		tia_values_lu=[tia_rk_lu, tia_lk_lu, tia_liver, tia_spleen, tia_bkg, tia_tumours_lu]
		dose_results_patient= ab_dose_177lu.main(patient, tia_values_lu, tumours[patient], true_dosimetry)

		mean_doses = {name: [vals[1]*7400/1000, vals[2]*7400/1000] for name, vals in dose_results_patient.items()}
		#mean_doses = {name: vals[1] for name, vals in dose_results_patient.items()}

		row = [mean_doses["right kidney"], mean_doses["left kidney"] ]
		row += [mean_doses[f"tumour {key}"] for key in tumours[patient]]   # lesions in your label order
		table.append(row)

		for name, vals in dose_results_patient.items():
			if not name.startswith("tumour ") or len(vals) < 4:
				continue
			dose = vals[1]
			volume = vals[3]
			if not np.isfinite(dose) or not np.isfinite(volume):
				continue

			all_lesions.append({"Patient": patient,"Lesion": name,"AD": dose * 7.4, "Volume": volume})
		print(f"Collected {len(all_lesions)} lesions across all patients.")

df = pd.DataFrame(table, index=list(patients.keys()))  
print(df.to_csv(sep="\t", header=False, index=False, float_format="%.4f"))

large_lesions = [l for l in all_lesions if l["Volume"] > 200]
print(f"{len(large_lesions)} lesions above 200 mL")
for l in sorted(large_lesions, key=lambda l: l["Volume"], reverse=True):
    print(f'{l["Patient"]} | {l["Lesion"]} | Volume = {l["Volume"]:.1f} mL | AD = {l["AD"]:.2f} Gy/GBq')

patient p11
performing dosimetry with the measured values
Collected 6 lesions across all patients.
patient p15
performing dosimetry with the measured values
Collected 17 lesions across all patients.
patient p24
performing dosimetry with the measured values
Collected 19 lesions across all patients.
patient p32
performing dosimetry with the measured values
Collected 24 lesions across all patients.
patient p33
performing dosimetry with the measured values
Collected 26 lesions across all patients.
patient p17
performing dosimetry with the measured values
Collected 34 lesions across all patients.
patient p23
performing dosimetry with the measured values
Collected 41 lesions across all patients.
patient p25
performing dosimetry with the measured values
Collected 43 lesions across all patients.
patient p28
performing dosimetry with the measured values
Collected 56 lesions across all patients.
patient p31
performing dosimetry with the measured values
Collected 66 lesions across all patients.
p

true dosimetry run

In [3]:
true_lk= {"p24": 57.1, "p25":95.2, "p33":59.0 ,"p15":68.8, "p23":88.3, "p17":23.5, "p32":25.9, "p28":47.8, "p31":53.8, "p13":40.5, "p12":49.0, "p11": 68.1565}
true_rk= {"p24": 60.3, "p25":116.1, "p33":50.1 , "p15":69.3, "p23":93.4, "p17":52.8, "p32":30.3, "p28":46.5, "p31":63.7, "p13":51.3, "p12":39.8, "p11":52.8016}

patients={"p11":0, "p15":1, "p24":2, "p32":3, "p33":4, "p17":5, "p23":6, "p25":7,  "p28":8, "p31":9, "p13":10, "p12":11}

tumours= {"p24":[11,12], "p25":[11, 12], "p15":[11, 12, 13, 14, 22, 23, 24, 25, 26, 27, 28], "p33":[11, 12], 
		   "p32":[11, 12, 13, 21, 22], "p23":[11, 12, 13, 14,22, 24, 25, 26], "p17":[11, 12, 13, 14, 22, 24, 25, 26],
		   "p31":[11, 12,13, 14, 22, 23, 24, 25, 26, 27], "p28":[11, 12, 13, 14, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31], 
           "p11":[23, 25, 26, 27, 28, 29], "p13":[11, 12], "p12":[11, 12, 13, 14]}

table = []
for patient in patients.keys():
		patient_id= patients[patient]
		tia_rk_lu, tia_lk_lu, tia_liver, tia_spleen, tia_bkg, tia_tumours_lu= tac_cumulated_activity.main(patient_id, true_dosimetry=True)

		#Lu-177 dose computation
		tia_values_lu=[tia_rk_lu, tia_lk_lu, tia_liver, tia_spleen, tia_bkg, tia_tumours_lu]
		dose_results_patient= ab_dose_177lu.main(patient, tia_values_lu, tumours[patient], true_dosimetry=True)

		mean_doses = {name: vals[1]*7400/1000 for name, vals in dose_results_patient.items()}
		#mean_doses = {name: vals[1] for name, vals in dose_results_patient.items()}
		
		row = [mean_doses["right kidney"], mean_doses["left kidney"]]
		row += [mean_doses[f"tumour {key}"] for key in tumours[patient]]   # lesions in your label order
		table.append(row)

df = pd.DataFrame(table, index=list(patients.keys()))   
print(df.to_csv(sep="\t", header=False, index=True, float_format="%.4f"))

patient p11
administered activity: 6546.4 MBq, scale factor = 1.130
performing dosimetry with the ground truth values
patient p15
administered activity: 6647.975 MBq, scale factor = 1.113
performing dosimetry with the ground truth values
patient p24
administered activity: 6629.0 MBq, scale factor = 1.116
performing dosimetry with the ground truth values
patient p32
administered activity: 7137.0 MBq, scale factor = 1.037
performing dosimetry with the ground truth values
patient p33
administered activity: 7140.0 MBq, scale factor = 1.036
performing dosimetry with the ground truth values
patient p17
administered activity: 6629.0 MBq, scale factor = 1.116
performing dosimetry with the ground truth values
patient p23
administered activity: 6751.0 MBq, scale factor = 1.096
performing dosimetry with the ground truth values
patient p25
administered activity: 6621.0 MBq, scale factor = 1.118
performing dosimetry with the ground truth values
patient p28
administered activity: 6159.0 MBq, scale f